In [1]:
!pip show chromadb rank_bm25 sentence-transformers | grep -E "Name|Version"

Name: sentence-transformers
Version: 5.4.0


In [2]:
!pip install -q chromadb rank_bm25

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 783.2 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 49.4 MB/s eta 0:00:0000:01m0:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages th

In [3]:
import os
import shutil

src = '/kaggle/input/datasets/harshamin07/researchlens-data'
dst = '/kaggle/working/'

os.makedirs(dst, exist_ok=True)

for filename in os.listdir(src):
    full_src = os.path.join(src, filename)
    if os.path.isfile(full_src):
        shutil.copy(full_src, os.path.join(dst, filename))

print("Loaded from dataset:")
for f in os.listdir(dst):
    size_mb = os.path.getsize(os.path.join(dst, f)) / (1024 * 1024)
    print(f"  ✓ {f} — {size_mb:.2f} MB")

Loaded from dataset:
  ✓ papers_metadata.json — 0.15 MB
  ✓ .virtual_documents — 0.00 MB
  ✓ qasper_pairs.json — 0.06 MB
  ✓ parsed_papers.json — 7.29 MB


In [4]:
import json
import re
import pickle
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import numpy as np

In [5]:
with open('/kaggle/working/parsed_papers.json', 'r') as f:
    all_pages = json.load(f)

print(f"Total pages loaded: {len(all_pages)}")
print(f"Sample keys: {all_pages[0].keys()}")
print(f"Sample text (first 200 chars): {all_pages[0]['text'][:200]}")

Total pages loaded: 1892
Sample keys: dict_keys(['paper_id', 'title', 'page_num', 'text'])
Sample text (first 200 chars): ENHANCING HUMAN-LIKE RESPONSES IN LARGE LANGUAGE
MODELS∗
Ethem Ya˘gız Çalık
X: @Weyaxi
Hugging Face: huggingface.co/Weyaxi
ethemyagiz1@gmail.com
Talha Rüzgar Akku¸s
X: @qbert_ai
Hugging Face: huggingf


In [6]:
def clean_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'BIBREF\d+', '', text)
    text = re.sub(r'FIGREF\d+', '', text)
    text = re.sub(r'TABREF\d+', '', text)
    return text.strip()

all_pages_clean = []
for page in all_pages:
    cleaned = clean_text(page['text'])
    if len(cleaned) > 100:
        all_pages_clean.append({
            'paper_id': page['paper_id'],
            'title': page['title'],
            'page_num': page['page_num'],
            'text': cleaned
        })

print(f"Pages after cleaning: {len(all_pages_clean)}")
print(f"Sample cleaned text: {all_pages_clean[0]['text'][:200]}")

Pages after cleaning: 1892
Sample cleaned text: ENHANCING HUMAN-LIKE RESPONSES IN LARGE LANGUAGE MODELS∗ Ethem Ya˘gız Çalık X: @Weyaxi Hugging Face: huggingface.co/Weyaxi ethemyagiz1@gmail.com Talha Rüzgar Akku¸s X: @qbert_ai Hugging Face: huggingf


In [7]:
def fixed_chunk(text, paper_id, title, page_num, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    chunk_idx = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk_text = ' '.join(words[start:end])
        
        if len(chunk_text) > 100:
            chunks.append({
                'chunk_id': f"{paper_id}_p{page_num}_c{chunk_idx}",
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': chunk_text,
                'chunk_type': 'fixed'
            })
            chunk_idx += 1
        
        start += chunk_size - overlap
    
    return chunks

fixed_chunks = []
for page in all_pages_clean:
    chunks = fixed_chunk(
        page['text'],
        page['paper_id'],
        page['title'],
        page['page_num']
    )
    fixed_chunks.extend(chunks)

print(f"Total fixed chunks: {len(fixed_chunks)}")
print(f"Average chunk length (words): {np.mean([len(c['text'].split()) for c in fixed_chunks]):.0f}")

Total fixed chunks: 3205
Average chunk length (words): 343


In [8]:
def semantic_chunk(text, paper_id, title, page_num, max_words=400, min_words=50):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current_sentences = []
    current_word_count = 0
    chunk_idx = 0
    
    for sentence in sentences:
        word_count = len(sentence.split())
        
        if current_word_count + word_count > max_words and current_word_count >= min_words:
            chunk_text = ' '.join(current_sentences)
            chunks.append({
                'chunk_id': f"{paper_id}_p{page_num}_s{chunk_idx}",
                'paper_id': paper_id,
                'title': title,
                'page_num': page_num,
                'text': chunk_text,
                'chunk_type': 'semantic'
            })
            chunk_idx += 1
            current_sentences = [sentence]
            current_word_count = word_count
        else:
            current_sentences.append(sentence)
            current_word_count += word_count
    
    if current_sentences and current_word_count >= min_words:
        chunks.append({
            'chunk_id': f"{paper_id}_p{page_num}_s{chunk_idx}",
            'paper_id': paper_id,
            'title': title,
            'page_num': page_num,
            'text': ' '.join(current_sentences),
            'chunk_type': 'semantic'
        })
    
    return chunks

semantic_chunks = []
for page in all_pages_clean:
    chunks = semantic_chunk(
        page['text'],
        page['paper_id'],
        page['title'],
        page['page_num']
    )
    semantic_chunks.extend(chunks)

print(f"Total semantic chunks: {len(semantic_chunks)}")
print(f"Average chunk length (words): {np.mean([len(c['text'].split()) for c in semantic_chunks]):.0f}")

Total semantic chunks: 3427
Average chunk length (words): 301


In [9]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

test_embedding = embed_model.encode("test sentence")
print(f"Embedding model loaded")
print(f"Embedding dimension: {len(test_embedding)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded
Embedding dimension: 384


In [10]:
all_chunks = fixed_chunks + semantic_chunks
all_texts = [c['text'] for c in all_chunks]

print(f"Embedding {len(all_texts)} chunks total...")

embeddings = embed_model.encode(
    all_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

fixed_embeddings = embeddings[:len(fixed_chunks)]
semantic_embeddings = embeddings[len(fixed_chunks):]

print(f"Fixed chunk embeddings shape: {fixed_embeddings.shape}")
print(f"Semantic chunk embeddings shape: {semantic_embeddings.shape}")

Embedding 6632 chunks total...


Batches:   0%|          | 0/208 [00:00<?, ?it/s]

Fixed chunk embeddings shape: (3205, 384)
Semantic chunk embeddings shape: (3427, 384)


In [11]:
chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')

try:
    chroma_client.delete_collection("fixed_chunks")
    chroma_client.delete_collection("semantic_chunks")
except:
    pass

fixed_collection = chroma_client.create_collection(
    name="fixed_chunks",
    metadata={"hnsw:space": "cosine"}
)

semantic_collection = chroma_client.create_collection(
    name="semantic_chunks",
    metadata={"hnsw:space": "cosine"}
)

print("Collections created")

Collections created


In [12]:
batch_size = 500

for i in range(0, len(fixed_chunks), batch_size):
    batch_chunks = fixed_chunks[i:i+batch_size]
    batch_embeddings = fixed_embeddings[i:i+batch_size]
    
    fixed_collection.add(
        ids=[c['chunk_id'] for c in batch_chunks],
        documents=[c['text'] for c in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        metadatas=[{
            'paper_id': c['paper_id'],
            'title': c['title'],
            'page_num': c['page_num'],
            'chunk_type': c['chunk_type']
        } for c in batch_chunks]
    )
    print(f"Fixed: added batch {i//batch_size + 1}/{(len(fixed_chunks)-1)//batch_size + 1}")

print(f"\nFixed collection total: {fixed_collection.count()}")

Fixed: added batch 1/7
Fixed: added batch 2/7
Fixed: added batch 3/7
Fixed: added batch 4/7
Fixed: added batch 5/7
Fixed: added batch 6/7
Fixed: added batch 7/7

Fixed collection total: 3205


In [13]:
for i in range(0, len(semantic_chunks), batch_size):
    batch_chunks = semantic_chunks[i:i+batch_size]
    batch_embeddings = semantic_embeddings[i:i+batch_size]
    
    semantic_collection.add(
        ids=[c['chunk_id'] for c in batch_chunks],
        documents=[c['text'] for c in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        metadatas=[{
            'paper_id': c['paper_id'],
            'title': c['title'],
            'page_num': c['page_num'],
            'chunk_type': c['chunk_type']
        } for c in batch_chunks]
    )
    print(f"Semantic: added batch {i//batch_size + 1}/{(len(semantic_chunks)-1)//batch_size + 1}")

print(f"\nSemantic collection total: {semantic_collection.count()}")

Semantic: added batch 1/7
Semantic: added batch 2/7
Semantic: added batch 3/7
Semantic: added batch 4/7
Semantic: added batch 5/7
Semantic: added batch 6/7
Semantic: added batch 7/7

Semantic collection total: 3427


In [14]:
bm25_corpus = [c['text'].lower().split() for c in fixed_chunks]
bm25_index = BM25Okapi(bm25_corpus)

bm25_metadata = [{
    'chunk_id': c['chunk_id'],
    'paper_id': c['paper_id'],
    'title': c['title'],
    'page_num': c['page_num'],
    'text': c['text']
} for c in fixed_chunks]

print(f"BM25 index built on {len(bm25_corpus)} documents")
print(f"Vocabulary size: {len(bm25_index.idf)}")

BM25 index built on 3205 documents
Vocabulary size: 99444


In [15]:
test_query = "attention mechanism in transformers"
test_embedding = embed_model.encode(test_query).tolist()

chroma_results = fixed_collection.query(
    query_embeddings=[test_embedding],
    n_results=3,
    include=['documents', 'distances', 'metadatas']
)

print("ChromaDB top 3 results:")
for i, (doc, dist, meta) in enumerate(zip(
    chroma_results['documents'][0],
    chroma_results['distances'][0],
    chroma_results['metadatas'][0]
)):
    print(f"\n  [{i+1}] Score: {1-dist:.3f} | Paper: {meta['title'][:60]}")
    print(f"       Text: {doc[:150]}")

bm25_scores = bm25_index.get_scores(test_query.lower().split())
top_bm25_idx = np.argsort(bm25_scores)[::-1][:3]

print("\n\nBM25 top 3 results:")
for i, idx in enumerate(top_bm25_idx):
    print(f"\n  [{i+1}] Score: {bm25_scores[idx]:.3f} | Paper: {bm25_metadata[idx]['title'][:60]}")
    print(f"       Text: {bm25_metadata[idx]['text'][:150]}")

ChromaDB top 3 results:

  [1] Score: 0.581 | Paper: A Survey on Transformer Compression
       Text: Advances in Neural Information Processing Systems, 2021. C.-F. Chen et al. Regionvit: Regional-to-local attention for vision transformers. arXiv prepr

  [2] Score: 0.570 | Paper: Benign Overfitting in Token Selection of Attention Mechanism
       Text: Benign Overfitting in Token Selection of Attention Mechanism LeCun, Y., Cortes, C., and Burges, C. Mnist handwritten digit database. ATT Labs [Online]

  [3] Score: 0.542 | Paper: Benign Overfitting in Token Selection of Attention Mechanism
       Text: Benign Overfitting in Token Selection of Attention Mechanism Keitaro Sakamoto 1 Issei Sato 1 Abstract Attention mechanism is a fundamental component o


BM25 top 3 results:

  [1] Score: 14.119 | Paper: Theoretical Insights into Fine-Tuning Attention Mechanism: G
       Text: Wk as a single unit (Wqk = WqWT k ), however, the benefits of fine-tuning Wq&Wv alone have yet to be further clar

In [16]:
with open('/kaggle/working/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25_index, f)

with open('/kaggle/working/bm25_metadata.pkl', 'wb') as f:
    pickle.dump(bm25_metadata, f)

with open('/kaggle/working/fixed_chunks.pkl', 'wb') as f:
    pickle.dump(fixed_chunks, f)

with open('/kaggle/working/semantic_chunks.pkl', 'wb') as f:
    pickle.dump(semantic_chunks, f)

print("Saved:")
files = ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl', 'semantic_chunks.pkl']
for fname in files:
    size_mb = os.path.getsize(f'/kaggle/working/{fname}') / (1024*1024)
    print(f"  ✓ {fname} — {size_mb:.2f} MB")

chroma_db_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, dirs, files_list in os.walk('/kaggle/working/chroma_db')
    for file in files_list
) / (1024*1024)
print(f"  ✓ chroma_db/ — {chroma_db_size:.2f} MB")

Saved:
  ✓ bm25_index.pkl — 8.52 MB
  ✓ bm25_metadata.pkl — 7.31 MB
  ✓ fixed_chunks.pkl — 7.33 MB
  ✓ semantic_chunks.pkl — 6.91 MB
  ✓ chroma_db/ — 119.35 MB


In [17]:
print("=== Notebook 02 Complete ===\n")
print(f"Fixed chunks:    {len(fixed_chunks)}")
print(f"Semantic chunks: {len(semantic_chunks)}")
print(f"ChromaDB fixed:  {fixed_collection.count()}")
print(f"ChromaDB sem:    {semantic_collection.count()}")
print(f"BM25 docs:       {len(bm25_corpus)}")

=== Notebook 02 Complete ===

Fixed chunks:    3205
Semantic chunks: 3427
ChromaDB fixed:  3205
ChromaDB sem:    3427
BM25 docs:       3205


In [19]:
import zipfile

zip_path = '/kaggle/working/indexes.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl', 'semantic_chunks.pkl']:
        zf.write(f'/kaggle/working/{fname}', fname)
    
    for root, dirs, files in os.walk('/kaggle/working/chroma_db'):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, '/kaggle/working')
            zf.write(full_path, arcname)

size_mb = os.path.getsize(zip_path) / (1024*1024)
print(f"✓ indexes.zip created — {size_mb:.2f} MB")


✓ indexes.zip created — 74.48 MB
